In [1]:
import os
import duckdb
import pandas as pd

# ----------------------------
# Paths
# ----------------------------
PROCESSED_DIR = r"data\processed"

opt_file   = os.path.join(PROCESSED_DIR, "option_prices_small.parquet")
sec_file   = os.path.join(PROCESSED_DIR, "security_prices_small.parquet")
fundq_file = os.path.join(PROCESSED_DIR, "compustat_fundq_small.parquet")
ccm_file   = os.path.join(PROCESSED_DIR, "ccm_link_table_small.parquet")
name_file  = os.path.join(PROCESSED_DIR, "security_name_history_small.parquet")
crsp_file  = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small.parquet")

# ----------------------------
# Connect DuckDB
# ----------------------------
con = duckdb.connect()

In [2]:
# ----------------------------
# Helper function
# ----------------------------
def run_check(title, query, limit_df_rows=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    try:
        df = con.execute(query).df()
        if len(df) > limit_df_rows:
            print(df.head(limit_df_rows).to_string(index=False))
            print(f"\n[showing first {limit_df_rows} rows out of {len(df)}]")
        else:
            print(df.to_string(index=False))
    except Exception as e:
        print(f"ERROR: {e}")


# =====================================================================================
# CHECK 0: Do the files exist?
# =====================================================================================
print("\n" + "=" * 100)
print("CHECK 0: FILE EXISTENCE")
print("=" * 100)

for f in [opt_file, sec_file, fundq_file, ccm_file, name_file, crsp_file]:
    print(f"{f} -> {'FOUND' if os.path.exists(f) else 'MISSING'}")


# =====================================================================================
# CHECK 1: Row counts for each processed table
# =====================================================================================
run_check(
    "CHECK 1A: option_prices_small row count",
    f"SELECT COUNT(*) AS n_rows FROM read_parquet('{opt_file}')"
)

run_check(
    "CHECK 1B: security_prices_small row count",
    f"SELECT COUNT(*) AS n_rows FROM read_parquet('{sec_file}')"
)

run_check(
    "CHECK 1C: compustat_fundq_small row count",
    f"SELECT COUNT(*) AS n_rows FROM read_parquet('{fundq_file}')"
)

run_check(
    "CHECK 1D: ccm_link_table_small row count",
    f"SELECT COUNT(*) AS n_rows FROM read_parquet('{ccm_file}')"
)

run_check(
    "CHECK 1E: security_name_history_small row count",
    f"SELECT COUNT(*) AS n_rows FROM read_parquet('{name_file}')"
)

run_check(
    "CHECK 1F: crsp_daily_stock_small row count",
    f"SELECT COUNT(*) AS n_rows FROM read_parquet('{crsp_file}')"
)


# =====================================================================================
# CHECK 2: Inspect schema / columns
# =====================================================================================
run_check(
    "CHECK 2A: option_prices_small schema",
    f"DESCRIBE SELECT * FROM read_parquet('{opt_file}')"
)

run_check(
    "CHECK 2B: security_prices_small schema",
    f"DESCRIBE SELECT * FROM read_parquet('{sec_file}')"
)

run_check(
    "CHECK 2C: compustat_fundq_small schema",
    f"DESCRIBE SELECT * FROM read_parquet('{fundq_file}')"
)

run_check(
    "CHECK 2D: ccm_link_table_small schema",
    f"DESCRIBE SELECT * FROM read_parquet('{ccm_file}')"
)

run_check(
    "CHECK 2E: security_name_history_small schema",
    f"DESCRIBE SELECT * FROM read_parquet('{name_file}')"
)

run_check(
    "CHECK 2F: crsp_daily_stock_small schema",
    f"DESCRIBE SELECT * FROM read_parquet('{crsp_file}')"
)


# =====================================================================================
# CHECK 3: Quick samples from each table
# =====================================================================================
run_check(
    "CHECK 3A: Sample rows from option_prices_small",
    f"SELECT * FROM read_parquet('{opt_file}') LIMIT 10"
)

run_check(
    "CHECK 3B: Sample rows from compustat_fundq_small",
    f"SELECT * FROM read_parquet('{fundq_file}') LIMIT 10"
)

run_check(
    "CHECK 3C: Sample rows from security_name_history_small",
    f"SELECT * FROM read_parquet('{name_file}') LIMIT 10"
)

run_check(
    "CHECK 3D: Sample rows from ccm_link_table_small",
    f"SELECT * FROM read_parquet('{ccm_file}') LIMIT 10"
)

run_check(
    "CHECK 3E: Sample rows from crsp_daily_stock_small",
    f"SELECT * FROM read_parquet('{crsp_file}') LIMIT 10"
)


# =====================================================================================
# CHECK 4: Important identifier presence + null checks
# =====================================================================================
run_check(
    "CHECK 4A: option_prices_small key identifier null counts",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(secid) AS nonnull_secid,
        COUNT(ticker) AS nonnull_ticker,
        COUNT(cusip) AS nonnull_cusip,
        COUNT(optionid) AS nonnull_optionid,
        COUNT(date) AS nonnull_date,
        COUNT(exdate) AS nonnull_exdate
    FROM read_parquet('{opt_file}')
    """
)

run_check(
    "CHECK 4B: security_name_history_small key identifier null counts",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(secid) AS nonnull_secid,
        COUNT(ticker) AS nonnull_ticker,
        COUNT(cusip) AS nonnull_cusip,
        COUNT(effect_date) AS nonnull_effect_date
    FROM read_parquet('{name_file}')
    """
)

run_check(
    "CHECK 4C: compustat_fundq_small key identifier null counts",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(gvkey) AS nonnull_gvkey,
        COUNT(tic) AS nonnull_tic,
        COUNT(datadate) AS nonnull_datadate,
        COUNT(rdq) AS nonnull_rdq
    FROM read_parquet('{fundq_file}')
    """
)

run_check(
    "CHECK 4D: ccm_link_table_small key identifier null counts",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(gvkey) AS nonnull_gvkey,
        COUNT(LPERMNO) AS nonnull_lpermno,
        COUNT(LINKDT) AS nonnull_linkdt,
        COUNT(LINKENDDT) AS nonnull_linkenddt
    FROM read_parquet('{ccm_file}')
    """
)

run_check(
    "CHECK 4E: crsp_daily_stock_small key identifier null counts",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(permno) AS nonnull_permno,
        COUNT(date) AS nonnull_date,
        COUNT(ticker) AS nonnull_ticker,
        COUNT(ncusip) AS nonnull_ncusip,
        COUNT(prc) AS nonnull_prc,
        COUNT(openprc) AS nonnull_openprc
    FROM read_parquet('{crsp_file}')
    """
)


# =====================================================================================
# CHECK 5: Does compustat_fundq_small actually have earnings-event structure?
# =====================================================================================
run_check(
    "CHECK 5A: Is rdq present and how many distinct earnings dates are there?",
    f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT rdq) AS distinct_rdq,
        MIN(rdq) AS min_rdq,
        MAX(rdq) AS max_rdq
    FROM read_parquet('{fundq_file}')
    """
)

run_check(
    "CHECK 5B: Firms with the most rdq observations",
    f"""
    SELECT
        tic,
        gvkey,
        COUNT(*) AS n_earnings_rows,
        MIN(rdq) AS first_rdq,
        MAX(rdq) AS last_rdq
    FROM read_parquet('{fundq_file}')
    GROUP BY tic, gvkey
    ORDER BY n_earnings_rows DESC, tic
    LIMIT 20
    """
)

run_check(
    "CHECK 5C: Example earnings history for a few firms",
    f"""
    SELECT
        tic, gvkey, datadate, rdq, fyearq, fqtr
    FROM read_parquet('{fundq_file}')
    WHERE tic IN (
        SELECT tic
        FROM read_parquet('{fundq_file}')
        GROUP BY tic
        HAVING COUNT(*) >= 8
        ORDER BY tic
        LIMIT 3
    )
    ORDER BY tic, rdq
    LIMIT 50
    """
)

run_check(
    "CHECK 5D: Duplicate earnings rows by gvkey + rdq",
    f"""
    SELECT
        gvkey,
        rdq,
        COUNT(*) AS n_rows
    FROM read_parquet('{fundq_file}')
    GROUP BY gvkey, rdq
    HAVING COUNT(*) > 1
    ORDER BY n_rows DESC, gvkey, rdq
    LIMIT 20
    """
)


# =====================================================================================
# CHECK 6: Is strike_price scaled by 1000?
# =====================================================================================
run_check(
    "CHECK 6A: Strike price distribution",
    f"""
    SELECT
        MIN(strike_price) AS min_strike_raw,
        APPROX_QUANTILE(strike_price, 0.01) AS p01,
        APPROX_QUANTILE(strike_price, 0.05) AS p05,
        APPROX_QUANTILE(strike_price, 0.50) AS median_strike_raw,
        APPROX_QUANTILE(strike_price, 0.95) AS p95,
        APPROX_QUANTILE(strike_price, 0.99) AS p99,
        MAX(strike_price) AS max_strike_raw
    FROM read_parquet('{opt_file}')
    """
)

run_check(
    "CHECK 6B: Compare raw strike and strike/1000 on sample rows",
    f"""
    SELECT
        ticker,
        date,
        exdate,
        strike_price,
        strike_price / 1000.0 AS strike_div_1000,
        cp_flag,
        best_bid,
        best_offer
    FROM read_parquet('{opt_file}')
    WHERE strike_price IS NOT NULL
    LIMIT 20
    """
)


# =====================================================================================
# CHECK 7: Option pricing sanity checks
# =====================================================================================
run_check(
    "CHECK 7A: Bid/ask sanity",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        SUM(CASE WHEN best_bid IS NULL OR best_offer IS NULL THEN 1 ELSE 0 END) AS missing_bidask,
        SUM(CASE WHEN best_bid < 0 THEN 1 ELSE 0 END) AS negative_bid,
        SUM(CASE WHEN best_offer < 0 THEN 1 ELSE 0 END) AS negative_offer,
        SUM(CASE WHEN best_offer < best_bid THEN 1 ELSE 0 END) AS crossed_market_rows
    FROM read_parquet('{opt_file}')
    """
)

run_check(
    "CHECK 7B: Midpoint sanity",
    f"""
    SELECT
        MIN(midpoint) AS min_midpoint,
        APPROX_QUANTILE(midpoint, 0.01) AS p01_mid,
        APPROX_QUANTILE(midpoint, 0.50) AS median_mid,
        APPROX_QUANTILE(midpoint, 0.99) AS p99_mid,
        MAX(midpoint) AS max_midpoint
    FROM read_parquet('{opt_file}')
    """
)

run_check(
    "CHECK 7C: DTE sanity",
    f"""
    SELECT
        MIN(DTE) AS min_dte,
        APPROX_QUANTILE(DTE, 0.01) AS p01_dte,
        APPROX_QUANTILE(DTE, 0.50) AS median_dte,
        APPROX_QUANTILE(DTE, 0.99) AS p99_dte,
        MAX(DTE) AS max_dte
    FROM read_parquet('{opt_file}')
    """
)


# =====================================================================================
# CHECK 8: Security price table sanity
# =====================================================================================
run_check(
    "CHECK 8A: security_prices_small identifier coverage",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT secid) AS distinct_secid,
        COUNT(DISTINCT ticker) AS distinct_ticker,
        COUNT(DISTINCT cusip) AS distinct_cusip,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM read_parquet('{sec_file}')
    """
)

run_check(
    "CHECK 8B: security_prices_small nulls in pricing fields",
    f"""
    SELECT
        SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS missing_close,
        SUM(CASE WHEN open IS NULL THEN 1 ELSE 0 END) AS missing_open,
        SUM(CASE WHEN return IS NULL THEN 1 ELSE 0 END) AS missing_return,
        SUM(CASE WHEN shrout IS NULL THEN 1 ELSE 0 END) AS missing_shrout
    FROM read_parquet('{sec_file}')
    """
)


# =====================================================================================
# CHECK 9: CRSP daily table sanity
# =====================================================================================
run_check(
    "CHECK 9A: crsp_daily_stock_small coverage",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT permno) AS distinct_permno,
        COUNT(DISTINCT ticker) AS distinct_ticker,
        COUNT(DISTINCT ncusip) AS distinct_ncusip,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM read_parquet('{crsp_file}')
    """
)

run_check(
    "CHECK 9B: CRSP missingness in core pricing fields",
    f"""
    SELECT
        SUM(CASE WHEN prc IS NULL THEN 1 ELSE 0 END) AS missing_prc,
        SUM(CASE WHEN openprc IS NULL THEN 1 ELSE 0 END) AS missing_openprc,
        SUM(CASE WHEN ret IS NULL THEN 1 ELSE 0 END) AS missing_ret,
        SUM(CASE WHEN shrout IS NULL THEN 1 ELSE 0 END) AS missing_shrout
    FROM read_parquet('{crsp_file}')
    """
)


# =====================================================================================
# CHECK 10: CCM link quality
# =====================================================================================
run_check(
    "CHECK 10A: LINKTYPE / LINKPRIM values",
    f"""
    SELECT
        LINKTYPE,
        LINKPRIM,
        COUNT(*) AS n_rows
    FROM read_parquet('{ccm_file}')
    GROUP BY LINKTYPE, LINKPRIM
    ORDER BY n_rows DESC
    """
)

run_check(
    "CHECK 10B: Date range of CCM links",
    f"""
    SELECT
        MIN(LINKDT) AS min_linkdt,
        MAX(LINKDT) AS max_linkdt,
        MIN(LINKENDDT) AS min_linkenddt,
        MAX(LINKENDDT) AS max_linkenddt
    FROM read_parquet('{ccm_file}')
    """
)

run_check(
    "CHECK 10C: Duplicate gvkey-permno link rows",
    f"""
    SELECT
        gvkey,
        LPERMNO,
        COUNT(*) AS n_rows
    FROM read_parquet('{ccm_file}')
    GROUP BY gvkey, LPERMNO
    HAVING COUNT(*) > 1
    ORDER BY n_rows DESC, gvkey, LPERMNO
    LIMIT 20
    """
)


# =====================================================================================
# CHECK 11: Security name history sanity
# =====================================================================================
run_check(
    "CHECK 11A: security_name_history coverage",
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT secid) AS distinct_secid,
        COUNT(DISTINCT ticker) AS distinct_ticker,
        COUNT(DISTINCT cusip) AS distinct_cusip,
        MIN(effect_date) AS min_effect_date,
        MAX(effect_date) AS max_effect_date
    FROM read_parquet('{name_file}')
    """
)

run_check(
    "CHECK 11B: One secid mapping to many tickers/cusips?",
    f"""
    SELECT
        secid,
        COUNT(DISTINCT ticker) AS n_tickers,
        COUNT(DISTINCT cusip) AS n_cusips
    FROM read_parquet('{name_file}')
    GROUP BY secid
    HAVING COUNT(DISTINCT ticker) > 1 OR COUNT(DISTINCT cusip) > 1
    ORDER BY n_tickers DESC, n_cusips DESC, secid
    LIMIT 20
    """
)


# =====================================================================================
# CHECK 12: Can we probably link Compustat -> CCM -> CRSP?
# =====================================================================================
run_check(
    "CHECK 12A: How many fundq rows can link to CCM by gvkey?",
    f"""
    SELECT
        COUNT(*) AS fundq_rows,
        SUM(CASE WHEN c.gvkey IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_ccm_match
    FROM read_parquet('{fundq_file}') f
    LEFT JOIN read_parquet('{ccm_file}') c
      ON f.gvkey = c.gvkey
    """
)

run_check(
    "CHECK 12B: Example Compustat -> CCM links",
    f"""
    SELECT
        f.tic,
        f.gvkey,
        f.rdq,
        c.LPERMNO,
        c.LINKDT,
        c.LINKENDDT
    FROM read_parquet('{fundq_file}') f
    LEFT JOIN read_parquet('{ccm_file}') c
      ON f.gvkey = c.gvkey
    WHERE c.gvkey IS NOT NULL
    ORDER BY f.rdq
    LIMIT 20
    """
)


# =====================================================================================
# CHECK 13: Can we probably link OptionMetrics tables internally by secid?
# =====================================================================================
run_check(
    "CHECK 13A: Distinct secid counts across OptionMetrics tables",
    f"""
    SELECT 'option_prices_small' AS table_name, COUNT(DISTINCT secid) AS distinct_secid
    FROM read_parquet('{opt_file}')
    UNION ALL
    SELECT 'security_prices_small' AS table_name, COUNT(DISTINCT secid) AS distinct_secid
    FROM read_parquet('{sec_file}')
    UNION ALL
    SELECT 'security_name_history_small' AS table_name, COUNT(DISTINCT secid) AS distinct_secid
    FROM read_parquet('{name_file}')
    """
)

run_check(
    "CHECK 13B: option_prices_small rows that match security_prices_small on secid",
    f"""
    SELECT
        COUNT(*) AS option_rows_checked,
        SUM(CASE WHEN s.secid IS NOT NULL THEN 1 ELSE 0 END) AS option_rows_with_secid_match
    FROM (
        SELECT DISTINCT secid
        FROM read_parquet('{opt_file}')
    ) o
    LEFT JOIN (
        SELECT DISTINCT secid
        FROM read_parquet('{sec_file}')
    ) s
      ON o.secid = s.secid
    """
)

run_check(
    "CHECK 13C: option_prices_small rows that match security_name_history_small on secid",
    f"""
    SELECT
        COUNT(*) AS option_rows_checked,
        SUM(CASE WHEN n.secid IS NOT NULL THEN 1 ELSE 0 END) AS option_rows_with_name_match
    FROM (
        SELECT DISTINCT secid
        FROM read_parquet('{opt_file}')
    ) o
    LEFT JOIN (
        SELECT DISTINCT secid
        FROM read_parquet('{name_file}')
    ) n
      ON o.secid = n.secid
    """
)


# =====================================================================================
# CHECK 14: Basic date coverage alignment
# =====================================================================================
run_check(
    "CHECK 14A: Date ranges across core tables",
    f"""
    SELECT 'option_prices_small' AS table_name, MIN(date) AS min_date, MAX(date) AS max_date
    FROM read_parquet('{opt_file}')
    UNION ALL
    SELECT 'security_prices_small' AS table_name, MIN(date) AS min_date, MAX(date) AS max_date
    FROM read_parquet('{sec_file}')
    UNION ALL
    SELECT 'compustat_fundq_small' AS table_name, MIN(rdq) AS min_date, MAX(rdq) AS max_date
    FROM read_parquet('{fundq_file}')
    UNION ALL
    SELECT 'crsp_daily_stock_small' AS table_name, MIN(date) AS min_date, MAX(date) AS max_date
    FROM read_parquet('{crsp_file}')
    """
)

print("\nDone. Send me the outputs section by section, especially CHECKS 2, 4, 5, 6, 10, 12, 13, and 14.")


CHECK 0: FILE EXISTENCE
data\processed\option_prices_small.parquet -> FOUND
data\processed\security_prices_small.parquet -> FOUND
data\processed\compustat_fundq_small.parquet -> FOUND
data\processed\ccm_link_table_small.parquet -> FOUND
data\processed\security_name_history_small.parquet -> FOUND
data\processed\crsp_daily_stock_small.parquet -> FOUND

CHECK 1A: option_prices_small row count
  n_rows
63698542

CHECK 1B: security_prices_small row count
 n_rows
 939282

CHECK 1C: compustat_fundq_small row count
 n_rows
  15723

CHECK 1D: ccm_link_table_small row count
 n_rows
    552

CHECK 1E: security_name_history_small row count
 n_rows
   2862

CHECK 1F: crsp_daily_stock_small row count
 n_rows
 784448

CHECK 2A: option_prices_small schema
    column_name column_type null  key default extra
          secid      BIGINT  YES None    None  None
           date        DATE  YES None    None  None
         exdate        DATE  YES None    None  None
        cp_flag     VARCHAR  YES None    

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 min_strike_raw   p01   p05  median_strike_raw     p95     p99  max_strike_raw
           3000 14627 29329             126295 1324937 2958263         6500000

CHECK 6B: Compare raw strike and strike/1000 on sample rows
ticker       date     exdate  strike_price  strike_div_1000 cp_flag  best_bid  best_offer
   AFL 2018-01-02 2018-01-19         87500             87.5       C      1.19        1.34
   AFL 2018-01-02 2018-01-19         87500             87.5       P      0.66        0.73
   AFL 2018-01-02 2018-02-16         85000             85.0       C      3.50        3.95
   AFL 2018-01-02 2018-02-16         87500             87.5       C      1.96        2.05
   AFL 2018-01-02 2018-02-16         90000             90.0       C      0.79        0.87
   AFL 2018-01-02 2018-02-16         85000             85.0       P      0.69        0.80
   AFL 2018-01-02 2018-02-16         87500             87.5       P      1.46        1.67
   AFL 2018-01-02 2018-02-16         90000             90.0  

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 min_dte  p01_dte  median_dte  p99_dte  max_dte
       7        7          31      116      119

CHECK 8A: security_prices_small identifier coverage
 n_rows  distinct_secid  distinct_ticker  distinct_cusip   min_date   max_date
 939282             504              498             504 2018-01-02 2025-08-29

CHECK 8B: security_prices_small nulls in pricing fields
 missing_close  missing_open  missing_return  missing_shrout
           0.0           0.0            36.0             0.0

CHECK 9A: crsp_daily_stock_small coverage
 n_rows  distinct_permno  distinct_ticker  distinct_ncusip   min_date   max_date
 784448              462              484              496 2018-01-02 2024-12-31

CHECK 9B: CRSP missingness in core pricing fields
 missing_prc  missing_openprc  missing_ret  missing_shrout
       990.0           2989.0       1021.0             8.0

CHECK 10A: LINKTYPE / LINKPRIM values
LINKTYPE LINKPRIM  n_rows
      LC        P     391
      LU        P     161

CHECK 10B: Date range 

In [3]:
fundq_file = os.path.join(PROCESSED_DIR, "compustat_fundq_small.parquet")
ccm_file   = os.path.join(PROCESSED_DIR, "ccm_link_table_small.parquet")

# -----------------------------------------------------------------------------
# 1. Deduplicate Compustat earnings events
# Rule:
#   one row per gvkey + rdq
#   keep latest datadate first
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE fundq_dedup AS
SELECT
    gvkey,
    tic,
    datadate,
    rdq,
    fyearq,
    fqtr,
    epspxq,
    prccq,
    mkvaltq
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY gvkey, rdq
            ORDER BY datadate DESC, fqtr DESC, fyearq DESC
        ) AS rn
    FROM read_parquet('{fundq_file}')
    WHERE rdq IS NOT NULL
      AND rdq >= DATE '2018-01-01'
      AND rdq <= DATE '2024-12-31'
) t
WHERE rn = 1
""")

# -----------------------------------------------------------------------------
# 2. Link Compustat earnings to CCM using valid link dates
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_ccm AS
SELECT
    f.gvkey,
    f.tic,
    f.datadate,
    f.rdq,
    f.fyearq,
    f.fqtr,
    f.epspxq,
    f.prccq,
    f.mkvaltq,
    c.LPERMNO AS permno,
    c.LINKDT,
    c.LINKENDDT,
    c.LINKTYPE,
    c.LINKPRIM
FROM fundq_dedup f
LEFT JOIN read_parquet('{ccm_file}') c
    ON f.gvkey = c.gvkey
   AND f.rdq >= c.LINKDT
   AND (c.LINKENDDT IS NULL OR f.rdq <= c.LINKENDDT)
""")

# -----------------------------------------------------------------------------
# 3. Checks
# -----------------------------------------------------------------------------
def show(title, query, n=20):
    print("\n" + "="*100)
    print(title)
    print("="*100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

show(
    "A. fundq_dedup row count",
    "SELECT COUNT(*) AS n_rows FROM fundq_dedup"
)

show(
    "B. Did dedup remove duplicate gvkey+rdq rows?",
    """
    SELECT COUNT(*) AS duplicate_groups
    FROM (
        SELECT gvkey, rdq, COUNT(*) AS n
        FROM fundq_dedup
        GROUP BY gvkey, rdq
        HAVING COUNT(*) > 1
    ) x
    """
)

show(
    "C. earnings_ccm row count",
    "SELECT COUNT(*) AS n_rows FROM earnings_ccm"
)

show(
    "D. How many earnings events got a permno?",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN permno IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_permno,
        SUM(CASE WHEN permno IS NULL THEN 1 ELSE 0 END) AS rows_without_permno
    FROM earnings_ccm
    """
)

show(
    "E. Distinct earnings events vs distinct mapped events",
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT gvkey || '|' || CAST(rdq AS VARCHAR)) AS distinct_events,
        COUNT(DISTINCT CASE WHEN permno IS NOT NULL THEN gvkey || '|' || CAST(rdq AS VARCHAR) END) AS distinct_events_with_permno
    FROM earnings_ccm
    """
)

show(
    "F. Events mapping to multiple permnos",
    """
    SELECT
        gvkey,
        rdq,
        COUNT(DISTINCT permno) AS n_permnos
    FROM earnings_ccm
    WHERE permno IS NOT NULL
    GROUP BY gvkey, rdq
    HAVING COUNT(DISTINCT permno) > 1
    ORDER BY n_permnos DESC, gvkey, rdq
    LIMIT 20
    """
)

show(
    "G. Sample clean earnings->CCM links",
    """
    SELECT
        gvkey,
        tic,
        rdq,
        datadate,
        permno,
        LINKDT,
        LINKENDDT,
        LINKTYPE,
        LINKPRIM
    FROM earnings_ccm
    WHERE permno IS NOT NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)

show(
    "H. Sample earnings rows without permno",
    """
    SELECT
        gvkey,
        tic,
        rdq,
        datadate
    FROM earnings_ccm
    WHERE permno IS NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)


A. fundq_dedup row count
 n_rows
  13237

B. Did dedup remove duplicate gvkey+rdq rows?
 duplicate_groups
                0

C. earnings_ccm row count
 n_rows
  13237

D. How many earnings events got a permno?
 total_rows  rows_with_permno  rows_without_permno
      13237           12966.0                271.0

E. Distinct earnings events vs distinct mapped events
 total_rows  distinct_events  distinct_events_with_permno
      13237            13237                        12966

F. Events mapping to multiple permnos
Empty DataFrame
Columns: [gvkey, rdq, n_permnos]
Index: []

G. Sample clean earnings->CCM links
 gvkey  tic        rdq   datadate  permno     LINKDT LINKENDDT LINKTYPE LINKPRIM
117768 NVDA 2018-02-08 2018-01-31   86580 1999-01-22       NaT       LU        P
126554    A 2018-02-14 2018-01-31   87432 1999-11-18       NaT       LC        P
001704 AMAT 2018-02-14 2018-01-31   14702 1972-12-14       NaT       LU        P
020779 CSCO 2018-02-14 2018-01-31   76076 1990-02-16     

In [4]:
fundq_file = os.path.join(PROCESSED_DIR, "compustat_fundq_small.parquet")
ccm_file   = os.path.join(PROCESSED_DIR, "ccm_link_table_small.parquet")
crsp_file  = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small.parquet")

# -----------------------------------------------------------------------------
# Rebuild clean base tables
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE fundq_dedup AS
SELECT
    gvkey,
    tic,
    datadate,
    rdq,
    fyearq,
    fqtr,
    epspxq,
    prccq,
    mkvaltq
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY gvkey, rdq
            ORDER BY datadate DESC, fqtr DESC, fyearq DESC
        ) AS rn
    FROM read_parquet('{fundq_file}')
    WHERE rdq IS NOT NULL
      AND rdq >= DATE '2018-01-01'
      AND rdq <= DATE '2024-12-31'
) t
WHERE rn = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_ccm AS
SELECT
    f.gvkey,
    f.tic,
    f.datadate,
    f.rdq,
    f.fyearq,
    f.fqtr,
    f.epspxq,
    f.prccq,
    f.mkvaltq,
    c.LPERMNO AS permno
FROM fundq_dedup f
LEFT JOIN read_parquet('{ccm_file}') c
    ON f.gvkey = c.gvkey
   AND f.rdq >= c.LINKDT
   AND (c.LINKENDDT IS NULL OR f.rdq <= c.LINKENDDT)
""")

# -----------------------------------------------------------------------------
# Keep only mapped events and safe date range
# -----------------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_base AS
SELECT *
FROM earnings_ccm
WHERE permno IS NOT NULL
  AND rdq <= DATE '2024-12-30'
""")

# -----------------------------------------------------------------------------
# CRSP subset
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp AS
SELECT
    permno,
    date,
    TICKER AS crsp_ticker,
    NCUSIP AS ncusip,
    COMNAM AS comnam,
    prc,
    openprc,
    ret,
    vol,
    shrout
FROM read_parquet('{crsp_file}')
""")

# -----------------------------------------------------------------------------
# Previous trading day before earnings
# -----------------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE prev_trade AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.tic,
        e.rdq,
        e.datadate,
        e.permno,
        c.date AS prev_date,
        c.prc  AS prev_prc,
        c.openprc AS prev_openprc,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.date DESC
        ) AS rn
    FROM earnings_base e
    LEFT JOIN crsp c
      ON e.permno = c.permno
     AND c.date < e.rdq
) t
WHERE rn = 1
""")

# -----------------------------------------------------------------------------
# Same trading day as earnings date
# -----------------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE same_trade AS
SELECT
    e.gvkey,
    e.tic,
    e.rdq,
    e.permno,
    c.date AS same_date,
    c.prc AS same_prc,
    c.openprc AS same_openprc
FROM earnings_base e
LEFT JOIN crsp c
  ON e.permno = c.permno
 AND c.date = e.rdq
""")

# -----------------------------------------------------------------------------
# Next trading day after earnings
# -----------------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE next_trade AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.tic,
        e.rdq,
        e.permno,
        c.date AS next_date,
        c.prc  AS next_prc,
        c.openprc AS next_openprc,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.date ASC
        ) AS rn
    FROM earnings_base e
    LEFT JOIN crsp c
      ON e.permno = c.permno
     AND c.date > e.rdq
) t
WHERE rn = 1
""")

# -----------------------------------------------------------------------------
# Combine
# -----------------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_prices AS
SELECT
    e.gvkey,
    e.tic,
    e.datadate,
    e.rdq,
    e.fyearq,
    e.fqtr,
    e.permno,

    p.prev_date,
    p.prev_prc,
    p.prev_openprc,

    s.same_date,
    s.same_prc,
    s.same_openprc,

    n.next_date,
    n.next_prc,
    n.next_openprc

FROM earnings_base e
LEFT JOIN prev_trade p
  ON e.gvkey = p.gvkey AND e.rdq = p.rdq
LEFT JOIN same_trade s
  ON e.gvkey = s.gvkey AND e.rdq = s.rdq
LEFT JOIN next_trade n
  ON e.gvkey = n.gvkey AND e.rdq = n.rdq
""")

# -----------------------------------------------------------------------------
# Helper printer
# -----------------------------------------------------------------------------
def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

# -----------------------------------------------------------------------------
# Checks
# -----------------------------------------------------------------------------
show(
    "1. earnings_base row count",
    "SELECT COUNT(*) AS n_rows FROM earnings_base"
)

show(
    "2. earnings_prices row count",
    "SELECT COUNT(*) AS n_rows FROM earnings_prices"
)

show(
    "3. Missingness in prev / same / next trading prices",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN prev_date IS NULL THEN 1 ELSE 0 END) AS missing_prev_date,
        SUM(CASE WHEN prev_prc IS NULL THEN 1 ELSE 0 END) AS missing_prev_prc,
        SUM(CASE WHEN same_date IS NULL THEN 1 ELSE 0 END) AS missing_same_date,
        SUM(CASE WHEN same_prc IS NULL THEN 1 ELSE 0 END) AS missing_same_prc,
        SUM(CASE WHEN next_date IS NULL THEN 1 ELSE 0 END) AS missing_next_date,
        SUM(CASE WHEN next_prc IS NULL THEN 1 ELSE 0 END) AS missing_next_prc
    FROM earnings_prices
    """
)

show(
    "4. Gap between rdq and previous trading date",
    """
    SELECT
        MIN(rdq - prev_date) AS min_gap_prev,
        APPROX_QUANTILE(rdq - prev_date, 0.50) AS median_gap_prev,
        APPROX_QUANTILE(rdq - prev_date, 0.95) AS p95_gap_prev,
        MAX(rdq - prev_date) AS max_gap_prev
    FROM earnings_prices
    WHERE prev_date IS NOT NULL
    """
)

show(
    "5. Gap between rdq and next trading date",
    """
    SELECT
        MIN(next_date - rdq) AS min_gap_next,
        APPROX_QUANTILE(next_date - rdq, 0.50) AS median_gap_next,
        APPROX_QUANTILE(next_date - rdq, 0.95) AS p95_gap_next,
        MAX(next_date - rdq) AS max_gap_next
    FROM earnings_prices
    WHERE next_date IS NOT NULL
    """
)

show(
    "6. Sample earnings_prices rows",
    """
    SELECT
        tic,
        permno,
        rdq,
        prev_date,
        prev_prc,
        same_date,
        same_prc,
        next_date,
        next_prc,
        prev_openprc,
        same_openprc,
        next_openprc
    FROM earnings_prices
    ORDER BY rdq, tic
    LIMIT 30
    """
)

show(
    "7. Rows where same_date is missing but prev and next exist",
    """
    SELECT
        tic,
        permno,
        rdq,
        prev_date,
        prev_prc,
        same_date,
        next_date,
        next_prc
    FROM earnings_prices
    WHERE same_date IS NULL
      AND prev_date IS NOT NULL
      AND next_date IS NOT NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)

show(
    "8. Rows missing prev or next prices",
    """
    SELECT
        tic,
        permno,
        rdq,
        prev_date,
        prev_prc,
        next_date,
        next_prc
    FROM earnings_prices
    WHERE prev_prc IS NULL OR next_prc IS NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


1. earnings_base row count
 n_rows
  12966

2. earnings_prices row count
 n_rows
  12966

3. Missingness in prev / same / next trading prices
 total_rows  missing_prev_date  missing_prev_prc  missing_same_date  missing_same_prc  missing_next_date  missing_next_prc
      12966             1676.0            1678.0             1686.0            1688.0             1664.0            1666.0

4. Gap between rdq and previous trading date
 min_gap_prev  median_gap_prev  p95_gap_prev  max_gap_prev
            1                1             3             4

5. Gap between rdq and next trading date
 min_gap_next  median_gap_next  p95_gap_next  max_gap_next
            1                1             3           438

6. Sample earnings_prices rows
 tic  permno        rdq  prev_date  prev_prc  same_date  same_prc  next_date  next_prc  prev_openprc  same_openprc  next_openprc
NVDA   86580 2018-02-08 2018-02-07 228.80000 2018-02-08 217.52000 2018-02-09 232.08000        229.58     233.88000     238.250

In [5]:
fundq_file = os.path.join(PROCESSED_DIR, "compustat_fundq_small.parquet")
ccm_file   = os.path.join(PROCESSED_DIR, "ccm_link_table_small.parquet")
crsp_file  = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small.parquet")

# -----------------------------------------------------------------------------
# Rebuild clean event base
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE fundq_dedup AS
SELECT
    gvkey,
    tic,
    datadate,
    rdq,
    fyearq,
    fqtr
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY gvkey, rdq
            ORDER BY datadate DESC, fqtr DESC, fyearq DESC
        ) AS rn
    FROM read_parquet('{fundq_file}')
    WHERE rdq IS NOT NULL
      AND rdq >= DATE '2018-01-01'
      AND rdq <= DATE '2024-12-31'
) t
WHERE rn = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_base AS
SELECT
    f.gvkey,
    f.tic,
    f.datadate,
    f.rdq,
    c.LPERMNO AS permno
FROM fundq_dedup f
LEFT JOIN read_parquet('{ccm_file}') c
    ON f.gvkey = c.gvkey
   AND f.rdq >= c.LINKDT
   AND (c.LINKENDDT IS NULL OR f.rdq <= c.LINKENDDT)
WHERE c.LPERMNO IS NOT NULL
  AND f.rdq <= DATE '2024-12-30'
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp_permno_dates AS
SELECT
    permno,
    MIN(date) AS min_date,
    MAX(date) AS max_date,
    COUNT(*) AS n_rows
FROM read_parquet('{crsp_file}')
GROUP BY permno
""")

def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

# -----------------------------------------------------------------------------
# 1. Event permnos vs CRSP permnos
# -----------------------------------------------------------------------------
show(
    "1. Distinct permnos in earnings_base vs CRSP",
    f"""
    SELECT
        (SELECT COUNT(DISTINCT permno) FROM earnings_base) AS earnings_permnos,
        (SELECT COUNT(DISTINCT permno) FROM read_parquet('{crsp_file}')) AS crsp_permnos
    """
)

show(
    "2. Event permnos missing entirely from CRSP",
    """
    SELECT
        COUNT(*) AS n_missing_permnos
    FROM (
        SELECT DISTINCT e.permno
        FROM earnings_base e
        LEFT JOIN crsp_permno_dates c
          ON e.permno = c.permno
        WHERE c.permno IS NULL
    ) x
    """
)

show(
    "3. Which event permnos are missing entirely from CRSP?",
    """
    SELECT
        e.permno,
        MIN(e.tic) AS example_ticker,
        COUNT(*) AS n_events
    FROM earnings_base e
    LEFT JOIN crsp_permno_dates c
      ON e.permno = c.permno
    WHERE c.permno IS NULL
    GROUP BY e.permno
    ORDER BY n_events DESC, e.permno
    LIMIT 50
    """
)

# -----------------------------------------------------------------------------
# 2. Events where permno exists in CRSP but rdq is outside CRSP date coverage
# -----------------------------------------------------------------------------
show(
    "4. Events whose permno exists in CRSP, but rdq falls outside that permno's CRSP date range",
    """
    SELECT
        COUNT(*) AS n_events_outside_range
    FROM earnings_base e
    JOIN crsp_permno_dates c
      ON e.permno = c.permno
    WHERE e.rdq < c.min_date OR e.rdq > c.max_date
    """
)

show(
    "5. Sample events outside CRSP permno date range",
    """
    SELECT
        e.tic,
        e.permno,
        e.rdq,
        c.min_date,
        c.max_date,
        c.n_rows
    FROM earnings_base e
    JOIN crsp_permno_dates c
      ON e.permno = c.permno
    WHERE e.rdq < c.min_date OR e.rdq > c.max_date
    ORDER BY e.rdq, e.tic
    LIMIT 50
    """
)

# -----------------------------------------------------------------------------
# 3. Events with permno in CRSP and rdq inside date range
# -----------------------------------------------------------------------------
show(
    "6. Events with usable CRSP permno-date coverage in principle",
    """
    SELECT
        COUNT(*) AS n_events_in_range
    FROM earnings_base e
    JOIN crsp_permno_dates c
      ON e.permno = c.permno
    WHERE e.rdq BETWEEN c.min_date AND c.max_date
    """
)

# -----------------------------------------------------------------------------
# 4. Tickers most affected by missing CRSP coverage
# -----------------------------------------------------------------------------
show(
    "7. Tickers with most events missing entire CRSP permno coverage",
    """
    SELECT
        e.tic,
        COUNT(*) AS n_events_missing
    FROM earnings_base e
    LEFT JOIN crsp_permno_dates c
      ON e.permno = c.permno
    WHERE c.permno IS NULL
    GROUP BY e.tic
    ORDER BY n_events_missing DESC, e.tic
    LIMIT 50
    """
)

show(
    "8. Tickers with most events outside CRSP date range",
    """
    SELECT
        e.tic,
        COUNT(*) AS n_events_outside_range
    FROM earnings_base e
    JOIN crsp_permno_dates c
      ON e.permno = c.permno
    WHERE e.rdq < c.min_date OR e.rdq > c.max_date
    GROUP BY e.tic
    ORDER BY n_events_outside_range DESC, e.tic
    LIMIT 50
    """
)

# -----------------------------------------------------------------------------
# 5. Check specific weird examples from your output
# -----------------------------------------------------------------------------
show(
    "9. CRSP date coverage for specific suspicious permnos",
    """
    SELECT
        permno,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(*) AS n_rows
    FROM read_parquet('data\\processed\\crsp_daily_stock_small.parquet')
    WHERE permno IN (60097, 29946, 93002, 89071, 75154, 92108)
    GROUP BY permno
    ORDER BY permno
    """
)


1. Distinct permnos in earnings_base vs CRSP
 earnings_permnos  crsp_permnos
              496           462

2. Event permnos missing entirely from CRSP
 n_missing_permnos
                68

3. Which event permnos are missing entirely from CRSP?
 permno example_ticker  n_events
  29946           BF.B        28
  60097            MDT        28
  75154            CCL        28
  89071            ACN        28
  12084           NXPI        27
  12431             TT        27
  13103           APTV        27
  13586            PNR        27
  13760           NCLH        27
  14277            SLB        27
  14297           ALLE        27
  15826            XYZ        27
  16581           INVH        27
  17307           VICI        27
  19583            IVZ        27
  39917             WY        27
  45356            JCI        27
  45751           MRSH        27
  46703            HST        27
  58413            FRT        27

[showing first 20 rows out of 50]

4. Events whose permno

In [6]:
RAW_DIR = "data"   # change if your raw files are elsewhere

fundq_file = os.path.join(PROCESSED_DIR, "compustat_fundq_small.parquet")
ccm_file   = os.path.join(PROCESSED_DIR, "ccm_link_table_small.parquet")

# IMPORTANT: replace this with your actual raw CRSP daily file path
RAW_CRSP_FILE = os.path.join(RAW_DIR, "crsp_daily_stock.csv.gz")

new_crsp_file = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small_v2.parquet")

# -----------------------------------------------------------------------------
# 1. Build required permno list from earnings events
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE fundq_dedup AS
SELECT
    gvkey,
    tic,
    datadate,
    rdq,
    fyearq,
    fqtr
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY gvkey, rdq
            ORDER BY datadate DESC, fqtr DESC, fyearq DESC
        ) AS rn
    FROM read_parquet('{fundq_file}')
    WHERE rdq IS NOT NULL
      AND rdq >= DATE '2018-01-01'
      AND rdq <= DATE '2024-12-31'
) t
WHERE rn = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE needed_permnos AS
SELECT DISTINCT
    c.LPERMNO AS permno
FROM fundq_dedup f
JOIN read_parquet('{ccm_file}') c
    ON f.gvkey = c.gvkey
   AND f.rdq >= c.LINKDT
   AND (c.LINKENDDT IS NULL OR f.rdq <= c.LINKENDDT)
WHERE c.LPERMNO IS NOT NULL
""")

# -----------------------------------------------------------------------------
# 2. Inspect raw CRSP schema first
# -----------------------------------------------------------------------------
print("\nInspecting raw CRSP schema...")
print(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{RAW_CRSP_FILE}', compression='gzip')").df().to_string(index=False))

# -----------------------------------------------------------------------------
# 3. Create corrected CRSP parquet using needed permnos only
# Adjust column names if your raw file uses slightly different names
# -----------------------------------------------------------------------------
con.execute(f"""
COPY (
    SELECT
        PERMNO AS permno,
        CAST(date AS DATE) AS date,
        TICKER,
        NCUSIP,
        COMNAM,
        SHRCD AS shrcd,
        EXCHCD AS exchcd,
        SICCD AS siccd,
        PRC AS prc,
        RET AS ret_raw,
        TRY_CAST(NULLIF(RET, 'C') AS DOUBLE) AS ret,
        VOL AS vol,
        SHROUT AS shrout,
        OPENPRC AS openprc
    FROM read_csv_auto('{RAW_CRSP_FILE}', compression='gzip', all_varchar=true)
    WHERE TRY_CAST(PERMNO AS BIGINT) IN (SELECT permno FROM needed_permnos)
) TO '{new_crsp_file}' (FORMAT PARQUET)
""")

print(f"\nCreated: {new_crsp_file}")

# -----------------------------------------------------------------------------
# 4. Quick validation
# -----------------------------------------------------------------------------
def show(title, query):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    print(con.execute(query).df().to_string(index=False))

show(
    "Distinct permnos in needed_permnos",
    "SELECT COUNT(DISTINCT permno) AS n_needed_permnos FROM needed_permnos"
)

show(
    "Distinct permnos in corrected CRSP file",
    f"SELECT COUNT(DISTINCT permno) AS n_crsp_permnos FROM read_parquet('{new_crsp_file}')"
)

show(
    "Coverage dates in corrected CRSP file",
    f"""
    SELECT
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(*) AS n_rows
    FROM read_parquet('{new_crsp_file}')
    """
)

show(
    "Check suspicious permnos now",
    f"""
    SELECT
        permno,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(*) AS n_rows
    FROM read_parquet('{new_crsp_file}')
    WHERE permno IN (60097, 29946, 75154, 89071, 92108, 93002)
    GROUP BY permno
    ORDER BY permno
    """
)


Inspecting raw CRSP schema...
column_name column_type null  key default extra
     PERMNO      BIGINT  YES None    None  None
       date        DATE  YES None    None  None
   NAMEENDT        DATE  YES None    None  None
      SHRCD      BIGINT  YES None    None  None
     EXCHCD      BIGINT  YES None    None  None
      SICCD      BIGINT  YES None    None  None
     NCUSIP     VARCHAR  YES None    None  None
     TICKER     VARCHAR  YES None    None  None
     COMNAM     VARCHAR  YES None    None  None
     SHRCLS     VARCHAR  YES None    None  None
    TSYMBOL     VARCHAR  YES None    None  None
      NAICS      BIGINT  YES None    None  None
   PRIMEXCH     VARCHAR  YES None    None  None
    TRDSTAT     VARCHAR  YES None    None  None
    SECSTAT     VARCHAR  YES None    None  None
     PERMCO      BIGINT  YES None    None  None
     ISSUNO      BIGINT  YES None    None  None
      HEXCD      BIGINT  YES None    None  None
     HSICCD      BIGINT  YES None    None  None
      CUS

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Created: data\processed\crsp_daily_stock_small_v2.parquet

Distinct permnos in needed_permnos
 n_needed_permnos
              496

Distinct permnos in corrected CRSP file
 n_crsp_permnos
            486

Coverage dates in corrected CRSP file
  min_date   max_date  n_rows
2018-01-02 2024-12-31  830802

Check suspicious permnos now
permno   min_date   max_date  n_rows
 60097 2018-01-02 2024-12-31    1761
 75154 2018-01-02 2024-12-31    1761
 89071 2018-01-02 2024-12-31    1761
 92108 2018-01-02 2024-12-31    1768
 93002 2018-01-02 2024-12-31    1761


In [7]:
import duckdb
import os
import pandas as pd

# =============================================================================
# Paths
# =============================================================================
PROCESSED_DIR = r"data\processed"

fundq_file = os.path.join(PROCESSED_DIR, "compustat_fundq_small.parquet")
ccm_file   = os.path.join(PROCESSED_DIR, "ccm_link_table_small.parquet")
crsp_file  = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small_v2.parquet")

# Optional output file
earnings_prices_out = os.path.join(PROCESSED_DIR, "earnings_prices_v2.parquet")

# =============================================================================
# Connect
# =============================================================================
con = duckdb.connect()

# =============================================================================
# Helper printer
# =============================================================================
def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))


# =============================================================================
# STEP 1: Deduplicate Compustat earnings events
# one row per gvkey + rdq
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE fundq_dedup AS
SELECT
    gvkey,
    tic,
    datadate,
    rdq,
    fyearq,
    fqtr,
    epspxq,
    prccq,
    mkvaltq
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY gvkey, rdq
            ORDER BY datadate DESC, fqtr DESC, fyearq DESC
        ) AS rn
    FROM read_parquet('{fundq_file}')
    WHERE rdq IS NOT NULL
      AND rdq >= DATE '2018-01-01'
      AND rdq <= DATE '2024-12-31'
) t
WHERE rn = 1
""")

# =============================================================================
# STEP 2: Link to CCM using valid link dates
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_ccm AS
SELECT
    f.gvkey,
    f.tic,
    f.datadate,
    f.rdq,
    f.fyearq,
    f.fqtr,
    f.epspxq,
    f.prccq,
    f.mkvaltq,
    c.LPERMNO AS permno
FROM fundq_dedup f
LEFT JOIN read_parquet('{ccm_file}') c
    ON f.gvkey = c.gvkey
   AND f.rdq >= c.LINKDT
   AND (c.LINKENDDT IS NULL OR f.rdq <= c.LINKENDDT)
""")

# =============================================================================
# STEP 3: Keep only mapped events and safe date range
# rdq <= 2024-12-30 so next trading day can exist in CRSP
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_base AS
SELECT *
FROM earnings_ccm
WHERE permno IS NOT NULL
  AND rdq <= DATE '2024-12-30'
""")

# =============================================================================
# STEP 4: Load corrected CRSP subset
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp AS
SELECT
    permno,
    date,
    TICKER AS crsp_ticker,
    NCUSIP AS ncusip,
    COMNAM AS comnam,
    prc,
    openprc,
    ret,
    vol,
    shrout
FROM read_parquet('{crsp_file}')
""")

# =============================================================================
# STEP 5: Previous trading day before earnings
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE prev_trade AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.tic,
        e.rdq,
        e.datadate,
        e.permno,
        c.date AS prev_date,
        c.prc AS prev_prc,
        c.openprc AS prev_openprc,
        c.ret AS prev_ret,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.date DESC
        ) AS rn
    FROM earnings_base e
    LEFT JOIN crsp c
      ON e.permno = c.permno
     AND c.date < e.rdq
) t
WHERE rn = 1
""")

# =============================================================================
# STEP 6: Same trading day as earnings date
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE same_trade AS
SELECT
    e.gvkey,
    e.tic,
    e.rdq,
    e.permno,
    c.date AS same_date,
    c.prc AS same_prc,
    c.openprc AS same_openprc,
    c.ret AS same_ret
FROM earnings_base e
LEFT JOIN crsp c
  ON e.permno = c.permno
 AND c.date = e.rdq
""")

# =============================================================================
# STEP 7: Next trading day after earnings
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE next_trade AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.tic,
        e.rdq,
        e.permno,
        c.date AS next_date,
        c.prc AS next_prc,
        c.openprc AS next_openprc,
        c.ret AS next_ret,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.date ASC
        ) AS rn
    FROM earnings_base e
    LEFT JOIN crsp c
      ON e.permno = c.permno
     AND c.date > e.rdq
) t
WHERE rn = 1
""")

# =============================================================================
# STEP 8: Combine into final earnings_prices table
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_prices AS
SELECT
    e.gvkey,
    e.tic,
    e.datadate,
    e.rdq,
    e.fyearq,
    e.fqtr,
    e.epspxq,
    e.prccq,
    e.mkvaltq,
    e.permno,

    p.prev_date,
    p.prev_prc,
    p.prev_openprc,
    p.prev_ret,

    s.same_date,
    s.same_prc,
    s.same_openprc,
    s.same_ret,

    n.next_date,
    n.next_prc,
    n.next_openprc,
    n.next_ret

FROM earnings_base e
LEFT JOIN prev_trade p
  ON e.gvkey = p.gvkey AND e.rdq = p.rdq
LEFT JOIN same_trade s
  ON e.gvkey = s.gvkey AND e.rdq = s.rdq
LEFT JOIN next_trade n
  ON e.gvkey = n.gvkey AND e.rdq = n.rdq
""")

# =============================================================================
# STEP 9: Save to parquet
# =============================================================================
con.execute(f"""
COPY earnings_prices TO '{earnings_prices_out}' (FORMAT PARQUET)
""")

print(f"\nSaved corrected earnings price table to:\n{earnings_prices_out}")

# =============================================================================
# CHECKS
# =============================================================================
show(
    "1. earnings_base row count",
    "SELECT COUNT(*) AS n_rows FROM earnings_base"
)

show(
    "2. earnings_prices row count",
    "SELECT COUNT(*) AS n_rows FROM earnings_prices"
)

show(
    "3. Missingness in prev / same / next trading prices",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN prev_date IS NULL THEN 1 ELSE 0 END) AS missing_prev_date,
        SUM(CASE WHEN prev_prc IS NULL THEN 1 ELSE 0 END) AS missing_prev_prc,
        SUM(CASE WHEN same_date IS NULL THEN 1 ELSE 0 END) AS missing_same_date,
        SUM(CASE WHEN same_prc IS NULL THEN 1 ELSE 0 END) AS missing_same_prc,
        SUM(CASE WHEN next_date IS NULL THEN 1 ELSE 0 END) AS missing_next_date,
        SUM(CASE WHEN next_prc IS NULL THEN 1 ELSE 0 END) AS missing_next_prc
    FROM earnings_prices
    """
)

show(
    "4. Gap between rdq and previous trading date",
    """
    SELECT
        MIN(rdq - prev_date) AS min_gap_prev,
        APPROX_QUANTILE(rdq - prev_date, 0.50) AS median_gap_prev,
        APPROX_QUANTILE(rdq - prev_date, 0.95) AS p95_gap_prev,
        MAX(rdq - prev_date) AS max_gap_prev
    FROM earnings_prices
    WHERE prev_date IS NOT NULL
    """
)

show(
    "5. Gap between rdq and next trading date",
    """
    SELECT
        MIN(next_date - rdq) AS min_gap_next,
        APPROX_QUANTILE(next_date - rdq, 0.50) AS median_gap_next,
        APPROX_QUANTILE(next_date - rdq, 0.95) AS p95_gap_next,
        MAX(next_date - rdq) AS max_gap_next
    FROM earnings_prices
    WHERE next_date IS NOT NULL
    """
)

show(
    "6. Sample earnings_prices rows",
    """
    SELECT
        tic,
        permno,
        rdq,
        prev_date,
        prev_prc,
        same_date,
        same_prc,
        next_date,
        next_prc,
        prev_openprc,
        same_openprc,
        next_openprc
    FROM earnings_prices
    ORDER BY rdq, tic
    LIMIT 30
    """
)

show(
    "7. Rows where same_date is missing but prev and next exist",
    """
    SELECT
        tic,
        permno,
        rdq,
        prev_date,
        prev_prc,
        same_date,
        next_date,
        next_prc
    FROM earnings_prices
    WHERE same_date IS NULL
      AND prev_date IS NOT NULL
      AND next_date IS NOT NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)

show(
    "8. Rows missing prev or next prices",
    """
    SELECT
        tic,
        permno,
        rdq,
        prev_date,
        prev_prc,
        next_date,
        next_prc
    FROM earnings_prices
    WHERE prev_prc IS NULL OR next_prc IS NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Saved corrected earnings price table to:
data\processed\earnings_prices_v2.parquet

1. earnings_base row count
 n_rows
  12966

2. earnings_prices row count
 n_rows
  12966

3. Missingness in prev / same / next trading prices
 total_rows  missing_prev_date  missing_prev_prc  missing_same_date  missing_same_prc  missing_next_date  missing_next_prc
      12966              187.0             190.0              198.0             200.0              183.0             185.0

4. Gap between rdq and previous trading date
 min_gap_prev  median_gap_prev  p95_gap_prev  max_gap_prev
            1                1             3             4

5. Gap between rdq and next trading date
 min_gap_next  median_gap_next  p95_gap_next  max_gap_next
            1                1             3             4

6. Sample earnings_prices rows
 tic  permno        rdq  prev_date  prev_prc  same_date  same_prc  next_date  next_prc prev_openprc same_openprc next_openprc
NVDA   86580 2018-02-08 2018-02-07 228.80000 

In [8]:
import duckdb
import os

con = duckdb.connect()

PROCESSED_DIR = r"data\processed"

earnings_prices_file = os.path.join(PROCESSED_DIR, "earnings_prices_v2.parquet")
earnings_prices_clean_file = os.path.join(PROCESSED_DIR, "earnings_prices_clean.parquet")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_prices AS
SELECT *
FROM read_parquet('{earnings_prices_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_prices_clean AS
SELECT *
FROM earnings_prices
WHERE prev_prc IS NOT NULL
  AND next_prc IS NOT NULL
""")

con.execute(f"""
COPY earnings_prices_clean TO '{earnings_prices_clean_file}' (FORMAT PARQUET)
""")

print(f"Saved clean earnings price sample to: {earnings_prices_clean_file}")

def show(title, query):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    print(con.execute(query).df().to_string(index=False))

show(
    "1. Total rows before filter",
    "SELECT COUNT(*) AS n_rows FROM earnings_prices"
)

show(
    "2. Total rows after filter",
    "SELECT COUNT(*) AS n_rows FROM earnings_prices_clean"
)

show(
    "3. Rows dropped",
    """
    SELECT
        COUNT(*) AS dropped_rows
    FROM earnings_prices
    WHERE prev_prc IS NULL OR next_prc IS NULL
    """
)

Saved clean earnings price sample to: data\processed\earnings_prices_clean.parquet

1. Total rows before filter
 n_rows
  12966

2. Total rows after filter
 n_rows
  12776

3. Rows dropped
 dropped_rows
          190


In [9]:
import duckdb
import os

con = duckdb.connect()

PROCESSED_DIR = r"data\processed"

earnings_prices_clean_file = os.path.join(PROCESSED_DIR, "earnings_prices_clean.parquet")
name_file = os.path.join(PROCESSED_DIR, "security_name_history_small.parquet")
crsp_file = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small_v2.parquet")

# -----------------------------------------------------------------------------
# Load clean earnings events
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_prices_clean AS
SELECT *
FROM read_parquet('{earnings_prices_clean_file}')
""")

# -----------------------------------------------------------------------------
# Build CRSP cusip history by permno/date
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp_id_history AS
SELECT DISTINCT
    permno,
    date,
    NCUSIP AS ncusip,
    TICKER AS crsp_ticker
FROM read_parquet('{crsp_file}')
WHERE NCUSIP IS NOT NULL
""")

# -----------------------------------------------------------------------------
# For each event, get the CRSP identifier on the prev trading day
# -----------------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE event_crsp_id AS
SELECT
    e.gvkey,
    e.tic,
    e.rdq,
    e.permno,
    e.prev_date,
    c.ncusip,
    c.crsp_ticker
FROM earnings_prices_clean e
LEFT JOIN crsp_id_history c
  ON e.permno = c.permno
 AND e.prev_date = c.date
""")

# -----------------------------------------------------------------------------
# Load OptionMetrics name history
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_name_history AS
SELECT
    secid,
    ticker AS om_ticker,
    cusip AS om_cusip,
    effect_date
FROM read_parquet('{name_file}')
""")

# -----------------------------------------------------------------------------
# Candidate mapping by cusip with date awareness:
# choose latest effect_date <= prev_date
# -----------------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE event_secid_candidates AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.tic,
        e.rdq,
        e.permno,
        e.prev_date,
        e.ncusip,
        e.crsp_ticker,
        o.secid,
        o.om_ticker,
        o.om_cusip,
        o.effect_date,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY o.effect_date DESC, o.secid
        ) AS rn
    FROM event_crsp_id e
    LEFT JOIN om_name_history o
      ON e.ncusip = o.om_cusip
     AND o.effect_date <= e.prev_date
) t
WHERE rn = 1
""")

def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

show(
    "1. Clean earnings rows",
    "SELECT COUNT(*) AS n_rows FROM earnings_prices_clean"
)

show(
    "2. Event rows with CRSP ncusip on prev_date",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ncusip IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_ncusip
    FROM event_crsp_id
    """
)

show(
    "3. Event rows with mapped secid",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN secid IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_secid,
        SUM(CASE WHEN secid IS NULL THEN 1 ELSE 0 END) AS rows_without_secid
    FROM event_secid_candidates
    """
)

show(
    "4. Sample mapped events",
    """
    SELECT
        tic,
        crsp_ticker,
        permno,
        rdq,
        prev_date,
        ncusip,
        secid,
        om_ticker,
        om_cusip,
        effect_date
    FROM event_secid_candidates
    WHERE secid IS NOT NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)

show(
    "5. Sample unmapped events",
    """
    SELECT
        tic,
        crsp_ticker,
        permno,
        rdq,
        prev_date,
        ncusip
    FROM event_secid_candidates
    WHERE secid IS NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)


1. Clean earnings rows
 n_rows
  12776

2. Event rows with CRSP ncusip on prev_date
 total_rows  rows_with_ncusip
      12776           12776.0

3. Event rows with mapped secid
 total_rows  rows_with_secid  rows_without_secid
      12776          12456.0               320.0

4. Sample mapped events
 tic crsp_ticker  permno        rdq  prev_date   ncusip  secid om_ticker om_cusip effect_date
NVDA        NVDA   86580 2018-02-08 2018-02-07 67066G10 108321      NVDA 67066G10  2011-06-13
   A           A   87432 2018-02-14 2018-02-13 00846U10 101149         A 00846U10  2016-12-08
AMAT        AMAT   14702 2018-02-14 2018-02-13 03822210 101610      AMAT 03822210  2011-06-10
CSCO        CSCO   76076 2018-02-14 2018-02-13 17275R10 103042      CSCO 17275R10  2014-01-08
NTAP        NTAP   82598 2018-02-14 2018-02-13 64110D10 108055      NTAP 64110D10  2014-01-10
 CPB         CPB   25320 2018-02-16 2018-02-15 13442910 102660       CPB 13442910  2000-11-28
  DE          DE   19350 2018-02-16 2018-